# 📈 Analyse des Résultats — Groupe G02
**Problématique P02 : Régularisation et Généralisation**  
**Modèle : BERT-base-uncased | Dataset : IMDb (D01)**

⚠️ **Prérequis** : Exécuter `main_experiment.py` avant ce notebook.

Ce notebook analyse en détail les résultats :
- Résultats Optuna (top trials, importance des hyperparamètres)
- Comparaison des 5 configurations de régularisation
- Analyse du loss landscape et de la sharpness
- Conclusions sur P02

## 0. Imports et chargement des résultats

In [ ]:
import sys, os, json
sys.path.insert(0, '/content/G02_PROJET/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import RESULTS_DIR, FIGURES_DIR

print('✅ Imports OK')
print(f'Results : {RESULTS_DIR}')
print(f'Figures : {FIGURES_DIR}')

## 1. Chargement des résultats Optuna

In [ ]:
# Charger tous les trials
trials = []
for fname in sorted(os.listdir(RESULTS_DIR)):
    if fname.startswith('trial_') and fname.endswith('.json'):
        with open(os.path.join(RESULTS_DIR, fname)) as f:
            trials.append(json.load(f))

print(f'{len(trials)} trials chargés\n')

df = pd.DataFrame(trials)
df_sorted = df.sort_values('val_f1', ascending=False).head(10)
print(df_sorted[['trial','weight_decay','dropout','lr','val_f1','val_accuracy','train_accuracy','gap']].to_string(index=False))

## 2. Résumé final

In [ ]:
with open(os.path.join(RESULTS_DIR, 'final_summary.json')) as f:
    summary = json.load(f)

print('Résumé final :')
for k, v in summary.items():
    print(f'  {k:30s} : {v}')

## 3. Visualisation des figures générées

In [ ]:
from IPython.display import Image, display
import ipywidgets as widgets

figures = sorted([f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')])
print(f'{len(figures)} figures disponibles :\n')
for f in figures:
    print(f'  {f}')

In [ ]:
# Afficher toutes les figures
for fname in figures:
    print(f'\n── {fname} ──')
    display(Image(os.path.join(FIGURES_DIR, fname), width=800))

## 4. Analyse de l'importance des hyperparamètres

In [ ]:
if len(trials) > 0:
    df_corr = pd.DataFrame(trials)[['weight_decay','dropout','lr','val_f1','gap']]
    
    fig, ax = plt.subplots(figsize=(7, 5))
    corr = df_corr.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r',
                center=0, ax=ax, linewidths=0.5,
                xticklabels=corr.columns, yticklabels=corr.columns)
    ax.set_title('Corrélations entre hyperparamètres et métriques')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/analysis_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nCorrelation val_f1 avec :')
    print(corr['val_f1'].sort_values(ascending=False))

## 5. Tableau récapitulatif des 5 configurations

In [ ]:
with open(os.path.join(RESULTS_DIR, 'sharpness_results.json')) as f:
    sharpness_data = json.load(f)

rows = []
for cfg, data in sharpness_data.items():
    rows.append({
        'Configuration': cfg,
        'Val F1':        f"{data['val_f1']:.4f}",
        'Val Acc':       f"{data['val_acc']:.4f}",
        'Train Acc':     f"{data['train_acc']:.4f}",
        'Gap':           f"{data['generalization_gap']:.4f}",
        'Sharpness':     f"{data['sharpness']:.5f}",
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

## 6. Conclusions P02

Répondez ici à la question de recherche :  
**Comment le weight decay et le dropout affectent-ils la généralisation ?**

| Observation | Conclusion |
|---|---|
| WD optimal ≈ 1e-4 | Régularisation L2 modérée = meilleur équilibre |
| Dropout optimal ≈ 0.1 | Valeur par défaut BERT souvent suffisante |
| Sharpness ↓ avec combinaison WD+Drop | Minima plats → meilleure généralisation |
| Optuna trouve la meilleure combinaison | Interaction WD×Dropout complexe → Bayésien utile |

**Meilleure configuration** : Optuna Best (WD=8.3e-4, Drop=0.1, LR=2.7e-5)  
**Test Accuracy** : 86.10% | **Test F1** : 85.80%